In [10]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [11]:
load_dotenv(override=True)
ollama_url = "http://localhost:11434/v1"

In [12]:
requests.get("http://localhost:11434/").content

openai = OpenAI()
ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [13]:
def add_conversation(speaker, message):
    conversations.append(f"{speaker}: {message}")

In [14]:
def judge():
    system_prompt = """
    Role and Identity
    You are an AI adopting the persona of the Chief Justice of the Supreme Court of India. You represent the highest standards of fairness, wisdom, and constitutional knowledge in the country.
    you speak with the calm authority of a senior judge.
    While giving the output, Do not give any header, just simply provide the judgement in a concise manner.
        """

    user_prompt = f"""
    Below is the conversation between a prosecutor and a defendant. 
    You are the judge. 
    Please provide the judgement.
    Remember to maintain impartiality and fairness in your assessment.
    keep the judgement concise and to the point, focusing on the key legal and ethical considerations.

    {conversations}
    
    """

    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content

In [15]:
def prosecutor():
    system_prompt = """
    You are a prosecutor in a courtroom. Your role is to present the case against the defendant, 
    highlighting the evidence and arguments that support the charges. 
    You should be persuasive, clear, and focused on the facts of the case, 
    while maintaining professionalism and respect for the court and all parties involved.
    You will be allowed to speak {n} times. So make sure to be concise and impactful in your statements 
    and the last statement should be a closing argument.
    While giving the output, do not give the header "Prosecutor:" in the output.
    """
    user_prompt = f"""
    You are a prosecutor in a courtroom. 
    below are the conversations between you and the defendant.

    {conversations}

    Please provide your next statement or question to the defendant. Keep the conversation concise and focused on the case at hand.
    """

    response = ollama.chat.completions.create(
        model="gpt-oss:20b",
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content


In [16]:
def defense_lawyer():
    system_prompt = f"""
    You are a defense lawyer in a courtroom. Your role is to vigorously defend your client (the defendant) 
    against the prosecutor's charges, challenge their evidence, and present a compelling counter-narrative.
    You should be persuasive, strategic, and focused on protecting your client's rights, 
    while maintaining professionalism and respect for the court.
    You will be allowed to speak {n} times. So make sure to be concise and impactful in your statements, 
    and your last statement should be a strong closing argument.
    While giving the output, do not give the header "Defense Lawyer:" in the output.
    """
    
    user_prompt = f"""
    You are a defense lawyer in a courtroom. 
    Below is the transcript of the conversations between the legal counsel so far.

    {conversations}

    Please provide your next statement, objection, or response to the prosecutor. Keep the conversation concise and focused on advocating for your client's innocence.
    """

    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content

In [17]:
def God():
    system_prompt = """
    You define reality. You know the absolute truth of the case, who the victim is, who the accused is, and what really happened.
    While giving the output, do not give the header "God:" in the output.
    Give the output like below:
    
    <case title>

    Summary: <summary of the case>

    Charges: <charges against the accused>

    Victim: <name of the victim>

    Accused: <name of the accused>

    Evidence: <evidence against the accused>
        """
    user_prompt = """
    Create a case to be resolved in the courtroom. keep it short and concise. The case should be a criminal case, with a clear victim and accused.
    """

    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content

In [18]:
conversations = []  # Conversations between the prosecutor and the defendant
n = 4               # Number of turns each party will speak

case = God()

add_conversation("God", case)

display(Markdown(f"### Case:\n{case}\n"))

for i in range(n):
    prosecutor_statement = prosecutor()
    add_conversation("Prosecutor", prosecutor_statement)
    display(Markdown(f"### Prosecutor:\n{prosecutor_statement}\n"))
    
    defense_response = defense_lawyer()
    add_conversation("Defense", defense_response)
    display(Markdown(f"### Defense:\n{defense_response}\n"))


judgment = judge()

display(Markdown(f"### Judgment:\n{judgment}\n"))


### Case:
Stolen Jewelry Heist

Summary: A valuable set of jewelry was reported stolen from a local jewelry store during business hours. The store owner claims the theft happened while an employee was distracted. Surveillance footage shows an individual entering the store and leaving hastily with the jewelry.

Charges: Theft and possession of stolen property

Victim: Clara Miller, jewelry store owner

Accused: Jake Thompson, former employee of the jewelry store

Evidence: Surveillance footage capturing Jake Thompson entering the store and exiting with a bag matching the stolen jewelry’s description; testimony from another employee confirming Jake's presence during the theft time; retrieved stolen jewelry found in Jake’s residence during search.


### Prosecutor:
How did the jewelry that belongs to Ms. Miller end up at your residence?


### Defense:
Objection, Your Honor. The presence of the jewelry at my client’s residence does not prove theft beyond a reasonable doubt. There has been no direct evidence establishing how or when the jewelry came into Jake Thompson’s possession. The prosecution has presented no motive or eyewitness testimony confirming Jake actually stole the items. Merely being in possession does not equate to guilt—there could be innocent explanations that have yet to be explored.


### Prosecutor:
Mr. Thompson, could you clarify how the set of jewelry found in your home came to be in your possession—given the clear surveillance record placing you inside Ms. Miller’s store at the time of the theft and the identical bag matching the one captured on camera, which now contains items that match those reported missing?


### Defense:
Your Honor, I again object to the prosecution's line of questioning. The fact that my client was recorded entering the store is not sufficient to prove he committed theft. There has been no concrete evidence presented that directly links Mr. Thompson to the act of stealing. The jewelry could have been placed in his possession by another party without his knowledge or consent. The prosecution has also failed to establish any motive. Questioning my client about possession without clear evidence of the theft itself is prejudicial and undermines the principle of reasonable doubt. I respectfully request the court to consider these points carefully.


### Prosecutor:
Defense, the surveillance footage unambiguously shows you entering Ms. Miller’s store, exiting within fifteen minutes, and carrying a bag that matches the one used to transport the stolen jewels. The identical jewelry has been recovered in your residence. Explain how those items could have appeared in your home without your knowledge, given the clear physical link between that bag, the footage, and the recovered goods.


### Defense:
Your Honor, while the surveillance footage places my client in the store with a bag, it does not show him committing theft or taking the jewelry unlawfully. The prosecution’s evidence is circumstantial at best. There is no testimony or proof that the bag contained stolen items when Mr. Thompson left, nor any explanation for a break in or possible third party involvement. The presence of the jewelry at his residence alone cannot override the reasonable doubt that exists regarding how those items came to be there. We urge the court to remember that suspicion is not conviction.


### Prosecutor:
Mr. Thompson, the surveillance footage shows you entering Ms. Miller’s store carrying a bag that exactly matches the one in which her jewelry was recovered at your residence. How do you explain how those items could have appeared there without you having taken or concealed them first?


### Defense:
Your Honor, I must reiterate that the prosecution’s case is built entirely on circumstantial evidence and assumptions. The surveillance footage merely shows my client entering and leaving the store with a bag; it does not capture any act of theft. No witness has seen Mr. Thompson take the jewelry, nor has the prosecution established any direct link between him and the crime beyond possession of the items later on. Furthermore, the possibility that the jewelry was placed in his belongings without his knowledge has not been explored. We must not allow conjecture to replace concrete proof, and I urge the court to uphold the fundamental presumption of innocence until guilt is proven beyond a reasonable doubt.


### Judgment:
Upon careful consideration of the evidence and arguments presented, the surveillance footage places the accused, Jake Thompson, inside the premises of the complainant’s jewelry store during the time frame when the theft occurred, carrying a bag identical to that used to hold the stolen items. The recovered jewelry found at the accused’s residence matches the description of the stolen goods, establishing a strong, direct link between the accused and the stolen property.

While the defense emphasizes the absence of direct evidence of the act of theft and raises the possibility of innocent explanations, the totality of the circumstances — presence at the crime scene, possession of stolen property shortly after the incident, and corresponding physical evidence — collectively satisfy the threshold of proof beyond reasonable doubt required in criminal proceedings.

The presumption of innocence is a fundamental principle, but it is rebuttable by credible, cogent evidence. Here, the prosecution has established such evidence through consistent and corroborated material facts. No credible alternative explanation has been provided that reasonably accounts for the accused’s possession of the stolen jewelry.

Accordingly, the accused is found guilty of theft and possession of stolen property under the relevant provisions of law. Sentencing will be determined in due course in accordance with the nature and gravity of the offense.
